# keras_climate quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anas-rz/keras-climate/blob/main/examples/01_quickstart.ipynb)

`keras_climate` is a [Keras 3](https://keras.io/keras_3/) (TensorFlow / JAX / PyTorch backend)
framework of models for climate modeling and Earth observation, plus tooling
to port pretrained weights from the original reference implementations
(almost always PyTorch) onto the Keras equivalents.

This notebook covers the basics:

1. installing `keras_climate` and picking a backend
2. building a segmentation model and a forecasting model from scratch
3. training with the standard `model.fit()` API

See also:

- **[Load a pretrained model](https://colab.research.google.com/github/anas-rz/keras-climate/blob/main/examples/02_pretrained_inference.ipynb)** — run inference with real, publicly-hosted checkpoints
- **[Finetune a pretrained model](https://colab.research.google.com/github/anas-rz/keras-climate/blob/main/examples/03_finetune_pretrained_model.ipynb)** — finetune a pretrained Sentinel-2 backbone on EuroSAT
- the full docs at <https://anas-rz.github.io/keras-climate/>


## 1. Install

In [ ]:
!pip install -q "git+https://github.com/anas-rz/keras-climate.git"

## 2. Pick a Keras 3 backend

Colab ships TensorFlow by default, which Keras 3 uses automatically. You can
switch to JAX or PyTorch instead by setting `KERAS_BACKEND` **before**
`keras` is imported anywhere in the process.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # or "jax" / "torch"

import keras

print("Keras version:", keras.__version__)
print("Backend:", keras.backend.backend())

## 3. Build a segmentation model

Every model in `keras_climate` is a plain `keras.Model` builder function —
no custom training loop is imposed, so `model.compile()` / `model.fit()` /
`model.save()` all work exactly as they would for any other Keras model.

In [ ]:
from keras_climate.remote_sensing import UNet

# 4-band input (e.g. RGB + near-infrared), 5 land-cover classes
model = UNet(input_shape=(256, 256, 4), num_classes=5)
model.summary()

## 4. Build a forecasting model

In [ ]:
from keras_climate.forecasting import PatchTST
import numpy as np

forecaster = PatchTST(seq_len=336, pred_len=96, num_channels=7)

x = np.random.randn(2, 336, 7).astype("float32")
y = forecaster(x)
print("forecast shape:", y.shape)  # (batch, pred_len, num_channels)

## 5. Train like any other Keras model

This uses random data purely to demonstrate the plumbing — swap in a real
`keras_climate.datasets` loader (see the
[data guide](https://anas-rz.github.io/keras-climate/models/data/)) or your
own `tf.data`/NumPy pipeline for real training.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

x = np.random.randn(4, 256, 256, 4).astype("float32")
y = np.random.randint(0, 5, size=(4, 256, 256)).astype("int32")

model.fit(x, y, epochs=1, batch_size=2)

## Next steps

- **[Load a pretrained model](https://colab.research.google.com/github/anas-rz/keras-climate/blob/main/examples/02_pretrained_inference.ipynb)**
  — several models ship a loader that downloads a real, publicly hosted
  checkpoint and returns a model with pretrained weights already loaded.
- **[Finetune a pretrained model](https://colab.research.google.com/github/anas-rz/keras-climate/blob/main/examples/03_finetune_pretrained_model.ipynb)**
  — finetune a pretrained Sentinel-2 ResNet-50 backbone on EuroSAT.
- Full docs: <https://anas-rz.github.io/keras-climate/>
